# Ardook grant fit: similarity model baseline on synthetic data

Compares the models in README section 2 (Proposed Technical Approach) on a balanced synthetic dataset.

**Task:** given a grant description, predict **High / Moderate / Low** fit for Ardook.

**Method:** no model is trained. Each model gives a grant a score for how similar it is to a description of Ardook. Two cutoffs, chosen on the validation split, turn the score into a class. Final metrics come from the test split only.

**How to run (Colab):** Runtime → Run all. A GPU is optional; the models are small enough for CPU. No files to upload.

> These are synthetic-only results. They measure how well similarity matches our labeling rules, not how well it judges real grants.

In [1]:
%pip -q install "sentence-transformers>=3.0"

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, random, re, time, warnings
import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_recall_fscore_support
from sentence_transformers import CrossEncoder, SentenceTransformer

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"  # Apple MPS crashes on these models
print("device:", DEVICE)

LABELS = ["Low", "Moderate", "High"]
LABEL2ID = {l: i for i, l in enumerate(LABELS)}

/private/tmp/claude-501/-Users-giapdo-Desktop-Ardook-proj/a4319d0f-99c3-4529-8b9d-efc359734cfd/scratchpad/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1104 12:51:12.576000 20687 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


device: cpu


## 1. Synthetic data

Each grant is built from three attributes, and fixed rules assign its label:

| Attribute | `good` | `partial` | `bad` |
|---|---|---|---|
| **Theme** | core mission: women's employment, disability inclusion, caregivers, social enterprise | adjacent: generic women entrepreneurs, SMEs, green business | unrelated: AI, deep tech, biotech, academic research |
| **Geography** | Armenia, Eastern Partnership, worldwide, US | Georgia only (Ardook hasn't launched there yet) | other countries only (India, Kenya, …) |
| **Entity rule** | open to social enterprises / businesses | depends on an unknown fact (nonprofit-only, <5 years old, ≥51% women-owned) | hard blocker (deep-tech, university-led, must build AI) |

**Label rule:** any `bad` attribute → **Low**; all three `good` → **High**; otherwise → **Moderate**.

- **Balanced:** equal examples per class, split 80/10/10 as the README proposes.
- **Held-out wording:** the test split uses sentence templates, funder names and phrases that train and validation never contain.
- **Hard negatives:** e.g. an AI grant that "gives priority" to women's employment is still Low.
- **US rule:** US-eligible grants count as eligible, a team decision that `docs/ardook-profile.md` doesn't reflect yet. To change it, move the two US phrases in `GEOS`.

In [3]:
THEMES = {
    "good": [
        "economic empowerment of women", "employment for mothers of children with disabilities",
        "disability inclusion and inclusive employment", "support for family caregivers of people with disabilities",
        "flexible, home-based work for women", "social enterprises that create decent jobs for vulnerable groups",
        "livelihoods for single mothers", "women-led social enterprises with measurable social impact",
        "inclusive businesses that employ marginalized women", "reducing inequality through dignified employment (SDG 8 and SDG 10)",
        "economic independence for caregivers", "income generation for women who cannot work outside the home",
        "job creation for disadvantaged women through social entrepreneurship", "families raising children with special needs",
        "women's livelihoods and income security", "work-from-home opportunities for parents of disabled children",
    ],
    "partial": [
        "growth of small and medium-sized enterprises", "women entrepreneurs in any sector",
        "green and eco-friendly small businesses", "community development projects",
        "youth employment and skills training", "service-sector small businesses",
        "digitalization of micro-enterprises", "local economic development in rural regions",
        "early-stage startups with a social mission", "circular economy and waste reduction",
        "civil society capacity building", "tourism and hospitality businesses",
    ],
    "bad": [
        "artificial intelligence and machine learning research", "deep-tech hardware startups",
        "biomedical and clinical research", "space and satellite technology",
        "quantum computing", "agricultural biotechnology R&D",
        "cybersecurity tools", "fintech and blockchain platforms",
        "contemporary art and cultural heritage", "academic STEM research at universities",
        "generative AI products", "semiconductor design",
        "climate science modelling", "battery and energy storage technology",
    ],
}
GEOS = {
    "good": [
        "organizations based in Armenia", "applicants from Eastern Partnership countries (Armenia, Azerbaijan, Georgia, Moldova, Ukraine)",
        "applicants worldwide", "organizations operating in the South Caucasus",
        "social enterprises registered in Armenia", "applicants from any country",
        "organizations in EU Neighbourhood East countries", "US-based organizations",
        "organizations registered in the United States", "applicants from low- and middle-income countries",
        "Armenian and international organizations", "organizations based in Yerevan or other Armenian regions",
    ],
    "partial": [
        "organizations registered and operating in Georgia", "applicants based in Tbilisi",
        "Georgian social enterprises", "organizations active in the regions of Georgia (the country)",
    ],
    "bad": [
        "organizations based in India", "Kenyan-registered companies", "residents of Brazil",
        "Canadian citizens and permanent residents", "UK-registered charities", "organizations in the Philippines",
        "applicants based in Nigeria", "Australian small businesses", "organizations in Southeast Asia",
        "applicants from Latin America", "Indonesian startups", "residents of Ontario",
    ],
}
ENTITIES = {
    "good": [
        "both for-profit companies and nonprofits are eligible", "social enterprises of any legal form may apply",
        "registered businesses with at least one year of operations may apply", "small businesses with paying customers are eligible",
        "early-stage and growth-stage ventures are welcome", "women-led enterprises are encouraged to apply",
        "organizations with a proven, revenue-generating model are eligible", "any legally registered organization may apply",
    ],
    "partial": [
        "only registered nonprofits / NGOs may apply", "companies must be less than 5 years old",
        "businesses must be at least 51% owned by women", "applicants must have annual revenue below $250,000",
        "only legally registered charities are eligible", "the organization must have been registered after 2021",
    ],
    "bad": [
        "applicants must demonstrate deep-tech innovation at TRL 4 or above", "the lead applicant must be a university or research institute",
        "projects must build an AI or software product", "a PhD-level principal investigator is required",
        "applicants must hold patented technology", "only hardware startups with a working prototype qualify",
    ],
}
FUNDER_PREFIX = ["Northbridge", "Silver Linden", "Ararat Horizons", "Harbor Light", "Open Field", "Cedar & Stone",
                 "Kestrel", "Brightwater", "Sevan Valley", "Lumen", "Tidewell", "Granite Peak",
                 "Willowmere", "Amber Coast", "Highland Commons", "Juniper", "Starling", "Riverbend", "Copperleaf", "Meadowgate"]
FUNDER_SUFFIX = ["Foundation", "Fund", "Trust", "Initiative", "Partnership", "Alliance"]
TYPES = ["grant", "microgrant", "prize", "accelerator", "fellowship", "challenge", "award", "seed fund"]
FILLERS = ["Applications are reviewed on a rolling basis.", "Shortlisted applicants will be invited to an interview.",
           "Selected teams receive mentoring in addition to funding.", "Reporting is required every six months.",
           "Applications must be submitted in English.", "Joint applications from consortia are accepted.",
           "Information sessions are held online before the deadline.", "Previous grantees may reapply."]

# Document templates. {theme}/{geo}/{entity} are filled from the pools above.
TEMPLATES = [
    "The {funder} invites applications for its {program}. The program supports projects focused on {theme}. Eligible applicants: {geo}. Note that {entity}. Awards of up to {amount} are available. {filler}",
    "{program} ({type}). Amount: {amount}. Deadline: {deadline}. Core requirements: {theme}; {geo}; {entity}.",
    "Call for proposals: {program}. Focus area: {theme}. Who can apply: {geo}. Eligibility: {entity}. Funding: {amount}. Deadline: {deadline}.",
    "The {program} backs work on {theme}. The call is open to {geo}, and {entity}. {filler} {extra}",
    "{funder} announces {amount} in funding through its {type} for {theme}. Applicants must be {geo}. Important: {entity}. {extra}",
    "Opportunity: {program}. We fund {theme}. {filler} Open to {geo}; {entity}. Award size: {amount}.",
    # --- below this line: test-only templates ---
    "About the opportunity: {program} is a {type} for initiatives in {theme}. Geographic scope: {geo}. Eligibility notes: {entity}. Up to {amount} per award. {extra}",
    "Funding priorities: {theme}. Applications are accepted from {geo}. {entity_cap}. Grant size: {amount}. Closing date: {deadline}. {filler}",
]
N_TRAIN_TEMPLATES = 6

def split_pool(pool, split):
    # Every 4th phrase is held out for the test split so test wording is partly unseen.
    return [p for i, p in enumerate(pool) if (i % 4 == 3) == (split == "test")]

def fit_label(theme, geo, entity):
    keys = (theme, geo, entity)
    if "bad" in keys:
        return "Low"
    if keys == ("good", "good", "good"):
        return "High"
    return "Moderate"

def make_grant(rng, split, theme, geo, entity):
    pick = lambda pool: rng.choice(split_pool(pool, split))
    funder = f"{pick(FUNDER_PREFIX)} {rng.choice(FUNDER_SUFFIX)}"
    gtype = pick(TYPES)
    # Hard negatives / distractors: an extra theme mention that does not change the label.
    if theme == "bad" and rng.random() < 0.4:
        extra = f"Priority is given to teams that also address {pick(THEMES['good'])}."
    elif rng.random() < 0.3:
        extra = f"Projects related to {pick(THEMES['partial'])} are also welcome."
    else:
        extra = ""
    templates = TEMPLATES[:N_TRAIN_TEMPLATES] if split != "test" else TEMPLATES[N_TRAIN_TEMPLATES:]
    ent = pick(ENTITIES[entity])
    text = rng.choice(templates).format(
        funder=funder, program=f"{funder} {gtype.title()}", type=gtype,
        theme=pick(THEMES[theme]), geo=pick(GEOS[geo]), entity=ent, entity_cap=ent[0].upper() + ent[1:],
        amount=f"{rng.choice(['$', '€'])}{rng.choice([2, 5, 10, 15, 25, 50, 100, 250]) * 1000:,}",
        deadline=f"{rng.randint(1, 28)} {rng.choice(['January', 'March', 'May', 'July', 'September', 'November'])} {rng.choice([2026, 2027])}",
        filler=rng.choice(FILLERS), extra=extra)
    return re.sub(r"\s+", " ", text).strip()

def generate(n_per_class, split, seed, noise=0.0):
    rng = random.Random(seed)
    counts = {l: 0 for l in LABELS}
    rows, seen = [], set()
    while min(counts.values()) < n_per_class:
        attrs = [rng.choice(["good", "partial", "bad"]) for _ in range(3)]
        label = fit_label(*attrs)
        if counts[label] >= n_per_class:
            continue
        text = make_grant(rng, split, *attrs)
        if text in seen:
            continue
        seen.add(text)
        counts[label] += 1
        rows.append({"text": text, "label": label, "theme": attrs[0], "geo": attrs[1], "entity": attrs[2]})
    df = pd.DataFrame(rows)
    if noise:
        # Flip a fraction of labels to a neighbouring class (Low<->Moderate<->High).
        flip = rng.sample(range(len(df)), int(noise * len(df)))
        for i in flip:
            j = LABEL2ID[df.at[i, "label"]]
            df.at[i, "label"] = LABELS[1] if j != 1 else rng.choice([LABELS[0], LABELS[2]])
    df["label_id"] = df["label"].map(LABEL2ID)
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)

In [4]:
N_TRAIN, N_VAL, N_TEST = 800, 100, 100   # per class (80/10/10)
train_df = generate(N_TRAIN, "train", seed=1)
val_df = generate(N_VAL, "val", seed=2)
test_df = generate(N_TEST, "test", seed=3)
assert not set(train_df.text) & set(test_df.text)

os.makedirs("data", exist_ok=True)
pd.concat([train_df.assign(split="train"), val_df.assign(split="val"), test_df.assign(split="test")]).to_csv(
    "data/synthetic_grants.csv", index=False)

for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{name:5s} n={len(df):4d}", df.label.value_counts().to_dict())

train n=2400 {'Moderate': 800, 'Low': 800, 'High': 800}
val   n= 300 {'Low': 100, 'High': 100, 'Moderate': 100}
test  n= 300 {'Low': 100, 'High': 100, 'Moderate': 100}


In [5]:
pd.set_option("display.max_colwidth", 300)
test_df.groupby("label").head(2)[["label", "theme", "geo", "entity", "text"]]

,label,theme,geo,entity,text
0,Low,bad,partial,good,"Funding priorities: fintech and blockchain platforms. Applications are accepted from organizations active in the regions of Georgia (the country). Small businesses with paying customers are eligible. Grant size: $50,000. Closing date: 27 November 2026. Shortlisted applicants will be invited to a..."
1,High,good,good,good,About the opportunity: Granite Peak Fund Seed Fund is a seed fund for initiatives in support for family caregivers of people with disabilities. Geographic scope: organizations based in Yerevan or other Armenian regions. Eligibility notes: small businesses with paying customers are eligible. Up t...
2,Moderate,partial,partial,good,"Funding priorities: local economic development in rural regions. Applications are accepted from organizations active in the regions of Georgia (the country). Any legally registered organization may apply. Grant size: €10,000. Closing date: 4 May 2026. Applications are reviewed on a rolling basis."
3,High,good,good,good,About the opportunity: Meadowgate Fund Accelerator is a accelerator for initiatives in work-from-home opportunities for parents of disabled children. Geographic scope: organizations operating in the South Caucasus. Eligibility notes: small businesses with paying customers are eligible. Up to €10...
4,Low,good,bad,good,"Funding priorities: work-from-home opportunities for parents of disabled children. Applications are accepted from Canadian citizens and permanent residents. Any legally registered organization may apply. Grant size: $2,000. Closing date: 16 September 2027. Reporting is required every six months."
6,Moderate,partial,partial,partial,"About the opportunity: Juniper Alliance Accelerator is a accelerator for initiatives in community development projects. Geographic scope: organizations active in the regions of Georgia (the country). Eligibility notes: applicants must have annual revenue below $250,000. Up to $25,000 per award."


## 2. Ardook description

This is the text every grant is compared against. It is summarized from `docs/ardook-profile.md`, the source of truth; update both together.

In [6]:
ARDOOK_TEXT = (
    "Ardook is a revenue-generating, women-led social enterprise in Yerevan, Armenia, expanding to Tbilisi, Georgia. "
    "It provides sustainable, home-based employment for mothers of children with disabilities, many of them single mothers, "
    "through a laundry and ironing service. Its focus is women's economic empowerment, disability inclusion, caregivers, "
    "inclusive employment, decent work, and livelihoods."
)
# Bag of words: the profile's "Keywords for matching", plus the terms section 2 of the README asks us to tune.
ARDOOK_KEYWORDS = [
    "mother", "caregiver", "single mother", "disabilit", "disabled", "special needs", "inclusive", "inclusion",
    "home-based", "work-from-home", "flexible", "women", "empowerment", "decent", "dignified", "livelihood",
    "social enterprise", "social entrepreneurship", "laundry", "household", "armenia", "yerevan", "gyumri",
    "georgia", "tbilisi", "south caucasus", "eastern partnership", "sdg 8", "sdg 10", "eco-friendly",
]

## 3. Scoring and evaluation

Every model produces one similarity score per grant, where higher means more like Ardook. The two cutoffs that give the best macro-F1 on the validation split are then applied to the test split.

**Macro** precision, recall and F1 are unweighted averages over the three classes. The 95% interval on F1 comes from bootstrapping the test set, so models whose intervals overlap heavily are not clearly different.

In [7]:
results, predictions = [], {}

def to_class(scores, low_cut, high_cut):
    return np.where(scores >= high_cut, LABEL2ID["High"], np.where(scores >= low_cut, LABEL2ID["Moderate"], LABEL2ID["Low"]))

def tune_cutoffs(scores, y):
    cands = np.unique(np.quantile(scores, np.linspace(0.02, 0.98, 49)))
    best, best_f1 = (cands[0], cands[-1]), -1
    for i, lo in enumerate(cands):
        for hi in cands[i + 1:]:
            f = f1_score(y, to_class(scores, lo, hi), average="macro", zero_division=0)
            if f > best_f1:
                best, best_f1 = (lo, hi), f
    return best

def f1_interval(y_true, y_pred, n=1000):
    rng = np.random.default_rng(SEED)
    idx = rng.integers(0, len(y_true), (n, len(y_true)))
    vals = [f1_score(y_true[i], y_pred[i], average="macro", zero_division=0) for i in idx]
    return np.percentile(vals, [2.5, 97.5])

def record(name, kind, y_pred, seconds):
    y_true = test_df.label_id.values
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    lo, hi = f1_interval(y_true, y_pred)
    results.append({"model": name, "type": kind, "accuracy": accuracy_score(y_true, y_pred), "precision": p,
                    "recall": r, "f1": f, "f1_95ci": f"{lo:.3f}-{hi:.3f}", "seconds": round(seconds, 1)})
    predictions[name] = np.asarray(y_pred)
    print(f"{name:40s} acc={results[-1]['accuracy']:.3f} P={p:.3f} R={r:.3f} F1={f:.3f} [{lo:.3f}-{hi:.3f}] ({seconds:.0f}s)")

def run(name, kind, score_fn):
    t = time.time()
    val_s, test_s = score_fn(val_df.text.tolist()), score_fn(test_df.text.tolist())
    assert np.isfinite(val_s).all() and np.isfinite(test_s).all(), f"{name} returned NaN/inf scores"
    low_cut, high_cut = tune_cutoffs(val_s, val_df.label_id.values)
    record(name, kind, to_class(test_s, low_cut, high_cut), time.time() - t)

## 4. Models

In [8]:
# Floor: always predict one class. Balanced data gives 33% accuracy.
record("Majority class", "floor", np.full(len(test_df), train_df.label_id.mode()[0]), 0.0)

# Bag of words: how many Ardook keywords appear in the grant
def keyword_score(texts):
    return np.array([sum(k in t.lower() for k in ARDOOK_KEYWORDS) for t in texts], dtype=float)
run("Ardook bag of words", "baseline", keyword_score)

# TF-IDF with English stop words removed; word weights are learned from the training grants
tfidf = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), sublinear_tf=True).fit(train_df.text)
ardook_vec = tfidf.transform([ARDOOK_TEXT])
run("TF-IDF cosine", "baseline", lambda texts: (tfidf.transform(texts) @ ardook_vec.T).toarray().ravel())

Majority class                           acc=0.333 P=0.111 R=0.333 F1=0.167 [0.146-0.186] (0s)


Ardook bag of words                      acc=0.470 P=0.447 R=0.470 F1=0.454 [0.402-0.502] (0s)


TF-IDF cosine                            acc=0.547 P=0.586 R=0.547 F1=0.505 [0.445-0.564] (1s)


In [9]:
# Bi-encoders: embed Ardook and each grant separately, then compare with cosine similarity.
BI_ENCODERS = {  # model -> instruction added to the Ardook query
    "sentence-transformers/all-MiniLM-L6-v2": "",
    "BAAI/bge-small-en-v1.5": "Represent this sentence for searching relevant passages: ",
}
for name, instruction in BI_ENCODERS.items():
    model = SentenceTransformer(name, device=DEVICE)
    query = model.encode([instruction + ARDOOK_TEXT], normalize_embeddings=True)
    run(name.split("/")[-1], "bi-encoder",
        lambda texts: model.encode(texts, batch_size=64, normalize_embeddings=True) @ query[0])
    del model

# Cross-encoder: reads Ardook and the grant together as a pair.
ce = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device=DEVICE)
if DEVICE == "cpu":
    ce.model.double()  # 32-bit math returned NaN for this model on an M2 Mac CPU
run("ms-marco-MiniLM-L-6-v2", "cross-encoder",
    lambda texts: np.asarray(ce.predict([(ARDOOK_TEXT, t) for t in texts], batch_size=32)))
del ce

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8121.01it/s]

all-MiniLM-L6-v2                         acc=0.560 P=0.615 R=0.560 F1=0.548 [0.490-0.601] (2s)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7764.34it/s]

bge-small-en-v1.5                        acc=0.507 P=0.489 R=0.507 F1=0.496 [0.443-0.544] (5s)


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 7466.84it/s]

ms-marco-MiniLM-L-6-v2                   acc=0.493 P=0.499 R=0.493 F1=0.475 [0.417-0.528] (13s)


## 5. Results

In [10]:
res = pd.DataFrame(results).sort_values("f1", ascending=False).reset_index(drop=True)
os.makedirs("results", exist_ok=True)
res.to_csv("results/similarity_model_comparison.csv", index=False)
display(Markdown(res.round(3).to_markdown(index=False)))

| model                  | type          |   accuracy |   precision |   recall |    f1 | f1_95ci     |   seconds |
|:-----------------------|:--------------|-----------:|------------:|---------:|------:|:------------|----------:|
| all-MiniLM-L6-v2       | bi-encoder    |      0.56  |       0.615 |    0.56  | 0.548 | 0.490-0.601 |       2.5 |
| TF-IDF cosine          | baseline      |      0.547 |       0.586 |    0.547 | 0.505 | 0.445-0.564 |       0.6 |
| bge-small-en-v1.5      | bi-encoder    |      0.507 |       0.489 |    0.507 | 0.496 | 0.443-0.544 |       4.9 |
| ms-marco-MiniLM-L-6-v2 | cross-encoder |      0.493 |       0.499 |    0.493 | 0.475 | 0.417-0.528 |      13.3 |
| Ardook bag of words    | baseline      |      0.47  |       0.447 |    0.47  | 0.454 | 0.402-0.502 |       0   |
| Majority class         | floor         |      0.333 |       0.111 |    0.333 | 0.167 | 0.146-0.186 |       0   |

In [11]:
# Per-class scores and confusion matrix (rows = true class, columns = predicted) for the top model.
best = res.model[0]
y_true = test_df.label_id.values
print("Best model:", best, "\n")
print(classification_report(y_true, predictions[best], target_names=LABELS, digits=3, zero_division=0))
pd.DataFrame(confusion_matrix(y_true, predictions[best]), index=[f"true {l}" for l in LABELS], columns=[f"pred {l}" for l in LABELS])

Best model: all-MiniLM-L6-v2 

              precision    recall  f1-score   support

         Low      0.500     0.850     0.630       100
    Moderate      0.525     0.420     0.467       100
        High      0.820     0.410     0.547       100

    accuracy                          0.560       300
   macro avg      0.615     0.560     0.548       300
weighted avg      0.615     0.560     0.548       300



,pred Low,pred Moderate,pred High
true Low,85,13,2
true Moderate,51,42,7
true High,34,25,41


### Can similarity catch each reason a grant is Low?

A grant can be Low because of its theme, its geography, or its entity rule. The table shows the share of those grants each model correctly calls Low. Only grants with exactly one `bad` attribute are counted, so each row isolates one reason.

In [12]:
low = test_df[test_df.label == "Low"]
causes = {"theme": "only the theme is bad", "geo": "only the geography is bad", "entity": "only the entity rule is bad"}
rows = {}
for name in res.model:
    pred = pd.Series(predictions[name], index=test_df.index)
    row = {}
    for col, desc in causes.items():
        others = [c for c in causes if c != col]
        mask = (low[col] == "bad") & (low[others] != "bad").all(axis=1)
        row[f"{desc} (n={mask.sum()})"] = (pred[low[mask].index] == LABEL2ID["Low"]).mean()
    rows[name] = row
display(Markdown(pd.DataFrame(rows).T.round(3).to_markdown()))

|                        |   only the theme is bad (n=24) |   only the geography is bad (n=20) |   only the entity rule is bad (n=17) |
|:-----------------------|-------------------------------:|-----------------------------------:|-------------------------------------:|
| all-MiniLM-L6-v2       |                          0.833 |                               1    |                                0.412 |
| TF-IDF cosine          |                          0.417 |                               0.35 |                                0.176 |
| bge-small-en-v1.5      |                          0.625 |                               0.75 |                                0.235 |
| ms-marco-MiniLM-L-6-v2 |                          0.708 |                               0.9  |                                0.353 |
| Ardook bag of words    |                          0.792 |                               0.6  |                                0.412 |
| Majority class         |                          1     |                               1    |                                1     |

### Are US-eligible grants treated fairly?

The Ardook description mentions Armenia and Georgia, not the US, so similarity may penalize US-eligible grants even though the labels count them as eligible. The table shows the share of truly High grants each model calls High, split by geography. The test split's only US phrase is "US-based organizations".

In [13]:
high = test_df[test_df.label == "High"]
is_us = high.text.str.contains("US-based")
rows = {}
for name in res.model:
    pred = pd.Series(predictions[name], index=test_df.index)
    rows[name] = {f"US-eligible (n={is_us.sum()})": (pred[high[is_us].index] == LABEL2ID["High"]).mean(),
                  f"Other geography (n={(~is_us).sum()})": (pred[high[~is_us].index] == LABEL2ID["High"]).mean()}
display(Markdown(pd.DataFrame(rows).T.round(3).to_markdown()))

|                        |   US-eligible (n=32) |   Other geography (n=68) |
|:-----------------------|---------------------:|-------------------------:|
| all-MiniLM-L6-v2       |                0     |                    0.603 |
| TF-IDF cosine          |                1     |                    1     |
| bge-small-en-v1.5      |                0     |                    0.912 |
| ms-marco-MiniLM-L-6-v2 |                0.156 |                    0.324 |
| Ardook bag of words    |                0     |                    0.853 |
| Majority class         |                0     |                    0     |